# 05 - Fairness Analysis

Цель: проверить senior/junior share, концентрацию и баланс ролей.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)


Project root: /Users/andrey/Documents/projects/COMPASS-AI


In [2]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
predictions = pd.read_csv(REPORTS / "test_predictions.csv")

scored = predictions.merge(employees, on="employee_id", how="left")
display(scored.head())


,assignment_id,task_id,employee_id,success_label,success_probability,prediction,plane_user_id,name,role,grade,experience_years,primary_stack,skills,current_workload,active_tasks_count,avg_completion_speed,avg_quality_score,deadline_reliability,learning_goals,mentor_level,availability,timezone
0,ASN-000014,TASK-0197,EMP-025,1,0.746463,1,NaN,Анастасия Беляева,qa_engineer,middle,2.9,qa_automation,"{""API Design"": 3, ""CI/CD"": 2, ""Data Analysis"":...",0.747,5,0.725,0.687,0.884,"[""CI/CD"", ""Monitoring"", ""Python"", ""Testing""]",2,available,Europe/Berlin
1,ASN-000027,TASK-0122,EMP-007,1,0.916877,1,NaN,Ольга Волкова,backend_developer,middle,4.5,backend_python,"{""API Design"": 2, ""CI/CD"": 3, ""Data Analysis"":...",0.672,3,0.700,0.819,0.902,"[""FastAPI"", ""PostgreSQL"", ""Python"", ""documenta...",3,available,Europe/Berlin
2,ASN-000037,TASK-0576,EMP-029,0,0.595679,1,NaN,Софья Гусева,data_ml_engineer,middle,2.3,data_ml,"{""API Design"": 0, ""CI/CD"": 2, ""Data Analysis"":...",0.847,7,0.668,0.804,0.840,"[""Data Analysis"", ""PostgreSQL"", ""Testing""]",1,available,Europe/Berlin
3,ASN-000053,TASK-1930,EMP-013,1,0.825275,1,NaN,Ксения Лебедева,backend_developer,senior,7.8,backend_python,"{""API Design"": 4, ""CI/CD"": 2, ""Data Analysis"":...",0.824,6,0.529,0.871,0.825,"[""Python"", ""Security"", ""System Design""]",4,available,Europe/Berlin
4,ASN-000056,TASK-0009,EMP-011,1,0.913111,1,NaN,Полина Васильева,backend_developer,senior,5.8,backend_python,"{""API Design"": 4, ""CI/CD"": 3, ""Data Analysis"":...",0.525,4,0.939,1.000,0.951,"[""Monitoring"", ""Security"", ""documentation""]",3,available,Europe/Berlin


In [3]:
top_by_task = (
    scored
    .sort_values(
        ["task_id", "success_probability"],
        ascending=[True, False],
    )
    .groupby("task_id")
    .head(1)
)

top_employee_share = top_by_task["employee_id"].value_counts(
    normalize=True
)

fairness_summary = {
    "recommended_tasks": int(top_by_task["task_id"].nunique()),
    "senior_recommendation_share": float(
        (top_by_task["grade"] == "senior").mean()
    ),
    "junior_recommendation_share": float(
        (top_by_task["grade"] == "junior").mean()
    ),
    "top_employee_concentration": float(
        top_employee_share.head(1).iloc[0]
    ),
    "average_recommended_workload": float(
        top_by_task["current_workload"].mean()
    ),
}

display(fairness_summary)


{'recommended_tasks': 375,
 'senior_recommendation_share': 0.9066666666666666,
 'junior_recommendation_share': 0.0,
 'top_employee_concentration': 0.4826666666666667,
 'average_recommended_workload': 0.4831226666666667}

In [4]:
import plotly.express as px

grade_counts = top_by_task["grade"].value_counts().reset_index()
grade_counts.columns = ["grade", "count"]

fig = px.bar(grade_counts, x="grade", y="count")
fig.update_layout(title="Top recommendations by grade")
fig.show()
fig.write_html(FIGURES / "fairness_recommendations_by_grade.html")


In [5]:
import plotly.express as px

role_counts = top_by_task["role"].value_counts().reset_index()
role_counts.columns = ["role", "count"]

fig = px.bar(role_counts, x="role", y="count")
fig.update_layout(title="Top recommendations by role")
fig.show()
fig.write_html(FIGURES / "fairness_recommendations_by_role.html")


In [6]:
import plotly.express as px

employee_concentration = (
    top_by_task["name"]
    .value_counts()
    .head(10)
    .reset_index()
)
employee_concentration.columns = ["name", "top1_recommendations"]

fig = px.bar(
    employee_concentration,
    x="name",
    y="top1_recommendations",
)
fig.update_layout(title="Top employee concentration")
fig.show()
fig.write_html(FIGURES / "fairness_top_employee_concentration.html")


In [7]:
fairness_path = REPORTS / "fairness_summary.json"

with open(fairness_path, "w", encoding="utf-8") as file:
    json.dump(fairness_summary, file, ensure_ascii=False, indent=2)

print("saved", fairness_path)


saved /Users/andrey/Documents/projects/COMPASS-AI/reports/fairness_summary.json
